# Notebook 01: GPU Memory Hierarchy

## Why This Matters

Every performance bottleneck in deep learning can be traced back to memory. Model OOM errors, slow training, poor GPU utilization -- these all have root causes in memory access patterns. Understanding the GPU memory hierarchy (registers, caches, HBM, PCIe) is the difference between writing code that saturates your hardware and code that wastes 80% of it. This knowledge is what separates ML engineers who can reason about performance from those who can only add batch size changes.


In [ ]:
%pip install -q torch numpy matplotlib

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import time

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print("Ready.")

## 1. The GPU Memory Hierarchy

From fastest to slowest:

| Level | Size | Bandwidth | Latency | Who accesses it |
|-------|------|-----------|---------|-----------------|
| Registers | ~256 KB/SM | ~30 TB/s | 1 cycle | One thread |
| L1 cache / Shared memory | 32-128 KB/SM | ~10 TB/s | ~20 cycles | Thread block (warp) |
| L2 cache | 4-40 MB | ~3-5 TB/s | ~200 cycles | All SMs |
| HBM (global memory) | 40-80 GB | 1-3.35 TB/s | ~600 cycles | All SMs |
| PCIe (CPU RAM) | unlimited | 32-64 GB/s | ms | Via DMA |
| NVLink | -- | 600 GB/s | -- | GPU-to-GPU |

**The fundamental rule**: bandwidth decreases by ~10x at each level. Code that spills to HBM instead of staying in L1/L2 pays a ~3-10x penalty.

**Arithmetic intensity** (FLOPs per byte accessed) determines where you sit on the roofline:
- Low intensity = memory-bound (bandwidth limited)
- High intensity = compute-bound (FLOP rate limited)


In [ ]:
# Bandwidth estimation: measure effective bandwidth of different operations
import time

def benchmark(fn, *args, reps=100, warmup=10):
    for _ in range(warmup):
        _ = fn(*args)
    if device == 'cuda':
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(reps):
        _ = fn(*args)
    if device == 'cuda':
        torch.cuda.synchronize()
    return (time.perf_counter() - t0) / reps * 1000  # ms

N = 1024
A = torch.randn(N, N, device=device)
B = torch.randn(N, N, device=device)

# Elementwise (memory-bound)
elementwise_time = benchmark(torch.relu, A)
bytes_accessed_ew = A.numel() * A.element_size() * 2  # read + write
ew_bandwidth_GBs = bytes_accessed_ew / elementwise_time / 1e6

# Matmul (compute-bound at large N)
matmul_time = benchmark(torch.mm, A, B)
bytes_accessed_mm = (A.numel() + B.numel() + N*N) * A.element_size()
mm_flops = 2 * N**3
mm_bandwidth_GBs = bytes_accessed_mm / matmul_time / 1e6
mm_tflops = mm_flops / matmul_time / 1e9
mm_arithmetic_intensity = mm_flops / bytes_accessed_mm

print(f'=== N={N}x{N} ===')
print(f'Elementwise ReLU:  {elementwise_time:.4f} ms, eff BW: {ew_bandwidth_GBs:.1f} GB/s')
print(f'Matrix multiply:   {matmul_time:.4f} ms, eff BW: {mm_bandwidth_GBs:.1f} GB/s')
print(f'  matmul TFLOP/s:  {mm_tflops:.1f}')
print(f'  arithmetic intensity: {mm_arithmetic_intensity:.1f} FLOPs/byte')
print(f'\nReLU is MEMORY-BOUND (low arithmetic intensity)')
print(f'Matmul is COMPUTE-BOUND at this size (high arithmetic intensity)')

## 2. The Roofline Model

The roofline model tells you the **theoretical peak performance** given:
- Peak compute (FLOP/s)
- Peak bandwidth (bytes/s)
- The operation's arithmetic intensity (FLOPs/byte)

$$\text{Achievable GFLOP/s} = \min(\text{Peak FLOP/s},\ \text{Intensity} \times \text{Peak BW})$$

Operations to the left of the "ridge point" are memory-bound. To the right: compute-bound.

For an A100 GPU:
- Peak FP16: ~312 TFLOP/s
- HBM bandwidth: ~2 TB/s
- Ridge point: 312e12 / 2e12 = 156 FLOPs/byte

Attention (at long sequences) has low arithmetic intensity and falls to the left of the ridge.
Matmul (large matrices) has high arithmetic intensity and is compute-bound.


In [ ]:
# Roofline model visualization
fig, ax = plt.subplots(figsize=(10, 6))

# Theoretical numbers for A100 SXM4
peak_flops_tf = 312   # TFLOP/s (FP16 with sparsity)
peak_bw_tbs = 2.0     # TB/s
ridge_point = peak_flops_tf / peak_bw_tbs  # FLOPs/byte

intensities = np.logspace(-1, 3, 500)

# Roofline
memory_roof = peak_bw_tbs * intensities   # TB/s * FLOPs/byte = TFLOP/s
compute_roof = np.ones_like(intensities) * peak_flops_tf
achievable = np.minimum(memory_roof, compute_roof)

ax.loglog(intensities, achievable, 'b-', linewidth=2, label='Roofline')
ax.axvline(x=ridge_point, color='gray', linestyle='--', alpha=0.5, label=f'Ridge: {ridge_point:.0f} FLOPs/byte')

# Annotate common ML operations
operations = {
    'Elementwise (ReLU, exp)': (0.5, None, 'red'),
    'LayerNorm': (4, None, 'orange'),
    'Attention (seq=2048)': (15, None, 'purple'),
    'Attention (seq=128)': (50, None, 'green'),
    'Large MatMul (M=N=K=4096)': (200, None, 'blue'),
}

for name, (intensity, perf, color) in operations.items():
    achievable_perf = min(peak_bw_tbs * intensity, peak_flops_tf)
    ax.plot(intensity, achievable_perf, 'o', color=color, markersize=10, zorder=5)
    ax.annotate(name, xy=(intensity, achievable_perf),
               xytext=(intensity * 1.3, achievable_perf * 0.7),
               fontsize=8, color=color,
               arrowprops=dict(arrowstyle='->', color=color, lw=1.5))

ax.set_xlabel('Arithmetic Intensity (FLOPs/byte)', fontsize=12)
ax.set_ylabel('Performance (TFLOP/s)', fontsize=12)
ax.set_title('Roofline Model: A100 SXM4\n(left = memory-bound, right = compute-bound)', fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim(0.1, 1000)
ax.set_ylim(0.1, 500)

plt.tight_layout()
plt.savefig('roofline_model.png', dpi=100)
plt.show()

print(f'A100 ridge point: {ridge_point:.0f} FLOPs/byte')
print(f'Operations with intensity < {ridge_point:.0f} are memory-bound on A100')
print(f'Attention with seq=2048 has intensity ~15 FLOPs/byte -> memory-bound')

## 3. Coalesced vs Strided Memory Access

GPUs read memory in **cacheline-sized chunks** (128 bytes = 32 float32 values on A100). When 32 threads in a warp each access consecutive 4-byte floats, they form a **coalesced access** -- one 128-byte transaction serves all 32 threads.

When threads access non-consecutive memory (e.g., reading every 32nd element), each access requires a separate transaction -- **32x more bandwidth** consumed.

**Why this matters for ML**:
- Row-major tensors: reading a row is coalesced; reading a column is strided
- Transpose operations change access patterns
- Batch dimension should usually be the outer dimension


In [ ]:
# Coalesced vs strided access demonstration

M, N = 1024, 1024

A = torch.randn(M, N, device=device)

# Row sum: reads each row contiguously (coalesced)
row_sum_time = benchmark(lambda x: x.sum(dim=1), A)

# Column sum: reads every N-th element (strided, non-coalesced)
col_sum_time = benchmark(lambda x: x.sum(dim=0), A)

print(f'Row sum (coalesced):   {row_sum_time:.4f} ms')
print(f'Col sum (strided):     {col_sum_time:.4f} ms')
print(f'Ratio: {col_sum_time/row_sum_time:.2f}x slower for strided access')

# Batch size and memory layout
# (B, C, H, W) NCHW -- PyTorch default, efficient for convolution
# (B, H, W, C) NHWC -- TensorFlow default, efficient for some ops

batch, channels, height, width = 32, 64, 28, 28
nchw = torch.randn(batch, channels, height, width, device=device)
nhwc = nchw.permute(0, 2, 3, 1).contiguous()

# Channels-first reduction (NCHW)
nchw_time = benchmark(lambda x: x.mean(dim=1), nchw)
nhwc_time = benchmark(lambda x: x.mean(dim=-1), nhwc)

print(f'\nChannel mean NCHW: {nchw_time:.4f} ms')
print(f'Channel mean NHWC: {nhwc_time:.4f} ms')
print('(Memory layout affects performance even for same computation)')

## 4. Why Batch Size and Model Size Interact with Memory Bandwidth

The GPU has to load weights from HBM for every forward pass. For a model with P parameters:
- Small batch: each weight byte is used for only B examples -> arithmetic intensity low
- Large batch: each weight byte is used for B examples -> higher arithmetic intensity

**The weight reuse principle**: increasing batch size increases arithmetic intensity without changing the amount of data transferred (weights stay the same). This is why large batches often give better hardware utilization, up to the point where HBM is saturated.

For a linear layer of shape (in, out):
- Weights: in * out * 4 bytes (float32)
- FLOPs: 2 * B * in * out
- Arithmetic intensity: 2 * B * in * out / (in * out * 4) = B/2 FLOPs/byte

For B=1 (inference): 0.5 FLOPs/byte (strongly memory-bound)
For B=128: 64 FLOPs/byte (approaching ridge on A100)


In [ ]:
# Arithmetic intensity vs batch size
in_features, out_features = 1024, 1024
W = torch.randn(in_features, out_features, device=device, dtype=torch.float16)

batch_sizes = [1, 4, 16, 64, 256, 1024]
times_ms = []
intensities = []
tflops = []

for B in batch_sizes:
    x = torch.randn(B, in_features, device=device, dtype=torch.float16)
    t = benchmark(torch.mm, x, W, reps=200, warmup=20)
    times_ms.append(t)
    
    flops = 2 * B * in_features * out_features
    bytes_W = W.numel() * W.element_size()
    bytes_x = x.numel() * x.element_size()
    intensity = flops / (bytes_W + bytes_x)
    intensities.append(intensity)
    tflops.append(flops / t / 1e9)

print('Batch | Intensity (FLOPs/B) | TFLOP/s | Time (ms)')
print('-' * 55)
for B, intens, tflop, t in zip(batch_sizes, intensities, tflops, times_ms):
    print(f'{B:5d} | {intens:18.1f} | {tflop:7.1f} | {t:.4f}')

fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogx(batch_sizes, tflops, 'o-', color='steelblue', linewidth=2)
ax.set_xlabel('Batch size')
ax.set_ylabel('TFLOP/s')
ax.set_title('Linear layer throughput vs batch size\n(larger batch = better hardware utilization)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('batch_throughput.png', dpi=100)
plt.show()

## Summary

### Key Concepts

| Concept | Value | Why it matters |
|---------|-------|---------------|
| L1 bandwidth | ~10 TB/s | Registers + shared memory: fastest level |
| HBM bandwidth | 1-3.35 TB/s | Most operations bottleneck here |
| PCIe bandwidth | 32-64 GB/s | Data transfer CPU->GPU; avoid during training |
| NVLink bandwidth | 600 GB/s | GPU-to-GPU: key for multi-GPU gradient sync |
| A100 ridge point | ~156 FLOPs/byte | Operations below this are memory-bound |
| Attention intensity | ~15 FLOPs/byte | Why FlashAttention matters -- kernel fusion |

**Key insight**: FlashAttention's breakthrough was fusing the attention computation to stay in L1/L2 instead of reading/writing attention matrices to HBM. This moved attention from ~15 FLOPs/byte effective intensity to ~160 FLOPs/byte by reusing the same memory.

**Next**: `02_nccl_collectives.ipynb` -- how GPUs communicate gradients during distributed training.
